# Ingesting and transforming IOT sensors from Wind Turbinge using Delta Lake and Spark API

<img style="float: right" width="300px" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-2.png" />

In this notebook, we'll show you an alternative to Spark Declarative Pipelines: building an ingestion pipeline with the Spark API.

As you'll see, this implementation is lower level than the Spark Declarative Pipelines pipeline, and you'll have control over all the implementation details (handling checkpoints, data quality etc).

Lower level also means more power. Using Spark API, you'll have unlimited capabilities to ingest data in Batch or Streaming.

If you're unsure what to use, start with Spark Declarative Pipelines!

*Remember that Databricks workflow can be used to orchestrate a mix of Spark Declarative Pipelines pipeline with standard Spark pipeline.*

### Dataset:

As reminder, we have multiple data sources coming from different system:

* <strong>Turbine metadata</strong>: Turbine ID, location (1 row per turbine)
* <strong>Turbine sensor stream</strong>: Realtime streaming flow from wind turbine sensor (vibration, energy produced, speed etc)
* <strong>Turbine status</strong>: Historical turbine status based to analyse which part is faulty (used as label in our ML model)


Leveraging Spark and Delta Lake makes such an implementation easy.


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F01-Data-ingestion%2Fplain-spark-delta-pipeline%2F01.5-Delta-pipeline-spark-iot-turbine&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2F01-Data-ingestion%2Fplain-spark-delta-pipeline%2F01.5-Delta-pipeline-spark-iot-turbine&version=1">

In [0]:
%pip install mlflow==2.22.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/29.0 MB ? eta -:--:--
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/29.0 MB 51.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/29.0 MB 230.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━ 19.9/29.0 MB 258.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 27.5/29.0 MB 223.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 218.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.

In [0]:
%run ../../_resources/00-setup

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_iot_platform`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


## Building a Spark Data pipeline with Delta Lake

In this example, we'll implement a end 2 end pipeline consuming our IOT sources. We'll use the medaillon architecture but could build a star schema, data vault or any other modelisation.



This can be challenging with traditional systems due to the following:
 * Data quality issue
 * Running concurrent operation
 * Running DELETE/UPDATE/MERGE over files
 * Governance & schema evolution
 * Performance ingesting millions of small files on cloud buckets
 * Processing & analysing unstructured data (image, video...)
 * Switching between batch or streaming depending of your requirement...

## Solving these challenges with Delta Lake

<div style="float:left">

**What's Delta Lake? It's a new OSS standard to bring SQL Transactional database capabilities on top of parquet files!**

Used as a new Spark format, built on top of Spark API / SQL

* **ACID transactions** (Multiple writers can simultaneously modify a data set)
* **Full DML support** (UPDATE/DELETE/MERGE)
* **BATCH and STREAMING** support
* **Data quality** (expectatiosn, Schema Enforcement, Inference and Evolution)
* **TIME TRAVEL** (Look back on how data looked like in the past)
* **Performance boost** with ZOrder, data skipping and Caching, solves small files issue 
</div>


<img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-logo.png" style="height: 200px"/>

<br style="clear: both">

We'll incrementally load new data with the autoloader, enrich this information and then load a model from MLFlow to perform our predictive maintenance forecast.

This information will then be used to build our DBSQL dashboard to analyse current turbine farm and impact on stock.

Let'simplement the following flow: 
 
<div><img width="1100px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-spark-full.png"/></div>

*Note that we're including the ML model our [Data Scientist built](TODO) using Databricks AutoML to predict the churn.*

## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) 1/ Explore the dataset

Let's review the files being received

In [0]:
%sql LIST '/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data'

path,name,size,modification_time
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00000-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-782-1-c000.snappy.parquet,part-00000-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-782-1-c000.snappy.parquet,668441,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00001-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-783-1-c000.snappy.parquet,part-00001-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-783-1-c000.snappy.parquet,668506,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00002-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-784-1-c000.snappy.parquet,part-00002-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-784-1-c000.snappy.parquet,668367,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00003-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-785-1-c000.snappy.parquet,part-00003-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-785-1-c000.snappy.parquet,668496,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00004-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-786-1-c000.snappy.parquet,part-00004-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-786-1-c000.snappy.parquet,668482,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00005-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-787-1-c000.snappy.parquet,part-00005-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-787-1-c000.snappy.parquet,668477,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00006-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-788-1-c000.snappy.parquet,part-00006-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-788-1-c000.snappy.parquet,668401,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00007-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-789-1-c000.snappy.parquet,part-00007-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-789-1-c000.snappy.parquet,668535,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00008-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-790-1-c000.snappy.parquet,part-00008-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-790-1-c000.snappy.parquet,668476,1749848985000
/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data/part-00009-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-791-1-c000.snappy.parquet,part-00009-tid-2947338609678659548-efa27e8c-0bc8-4838-8d13-7b999aade6b2-791-1-c000.snappy.parquet,668308,1749848985000


In [0]:
%sql
SELECT * FROM PARQUET.`/Volumes/main/dbdemos_iot_platform/turbine_raw_landing/incoming_data`

timestamp,sensor_E,sensor_C,sensor_B,sensor_A,sensor_F,sensor_D,energy,turbine_id
1705427048,3.680238993760711,-2.4017933132799176,2.0734829853092407,-1.1915564556400149,2.0482410477610644,-1.9360463128179886,0.9304719240455787,74636c4f-bbdc-f311-5c57-35ea854368b8
1705426198,3.312927456066121,-3.2943904381650495,1.7129599873594428,-1.5422393155011818,-0.7113818292743428,-3.2025684973445037,0.3535372117456058,079c4b70-2dc1-c0fd-8d7c-5848a9bb9ca1
1705427628,3.1387600562150717,-0.5223851799291372,-4.1865976618062195,-1.133437190821339,0.20705152151896367,0.12016256275126547,0.2906999474631001,aa891d19-a1cc-765c-79a2-84641d0cfa3d
1705425258,-4.203939680141756,0.1872185111068283,-5.674889882508366,-1.4180791511545845,1.0796994995124913,3.3468962162759146,0.12286956044787721,610e4ad5-09c4-7055-dff4-948fe6b4f832
1705426548,0.9748443997729632,-0.022331386334189407,2.6310923396846304,-1.0619656930712502,0.2670447950523569,0.28720191435832376,0.1266601231499892,2e281650-1188-af2b-9bab-6dd900949476
1705425798,1.2175836704395588,2.060671440091906,0.06218858335929811,-0.05922187720398453,-0.1869436889194076,-1.0841291452456507,0.2588960972567357,aca129eb-c4ce-3b2e-809d-0048a10f47a9
1705426708,-0.6871497376437823,2.870935531569903,-0.5147379962024177,0.24181346185582409,-0.354497421506643,-0.4001582139744091,0.17513194913188726,177de88e-6b8a-d678-29b4-1769253357a6
1705427558,0.9889464457185069,2.2461851444229453,-1.5485414786757334,-1.3835714169352635,-2.072642210843528,-2.588021090654448,0.7127298715753021,375598f8-4414-f8ad-a904-174506a7418d
1705425388,2.546391011831106,-3.094224686903196,1.3736642384513544,-0.6439342519580967,-3.5614189701899637,-0.21666308018688618,0.29255703197561245,0e10452c-287e-096f-e23d-f9e44065c2ab
1705427538,1.6844202050811858,-2.0732665263393044,1.3606557116176927,-1.0088970294275112,-0.9539893877511418,-2.2764375923992315,0.028686180134204584,4fa9f715-e248-5263-586a-c6018e9f9c9a


### 1/ Loading our data using Databricks Autoloader (cloud_files)
<div style="float:right">
  <img width="700px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-spark-1.png"/>
</div>
  
Autoloader allow us to efficiently ingest millions of files from a cloud storage, and support efficient schema inference and evolution at scale.

For more details on autoloader, run `dbdemos.install('auto-loader')`

Let's use it to create our pipeline and ingest the raw JSON & CSV data being delivered in our blob storage `/demos/retail/churn/...`. 

In [0]:
%sql
-- Note: tables are automatically created during  .writeStream.table("sensor_bronze") operation, but we can also use plain SQL to create them:
CREATE TABLE IF NOT EXISTS spark_sensor_bronze (
  energy   DOUBLE,
  sensor_A DOUBLE,
  sensor_B DOUBLE,
  sensor_C DOUBLE,
  sensor_D DOUBLE,
  sensor_E DOUBLE,
  sensor_F DOUBLE,
  timestamp LONG,
  turbine_id STRING     
  ) using delta 
    CLUSTER BY (turbine_id) -- Requests by turbine ID will be faster, Databricks manage the file layout for you out of the box. 
    TBLPROPERTIES (
     delta.autooptimize.optimizewrite = TRUE,
     delta.autooptimize.autocompact   = TRUE ); 
-- With these 2 last options, Databricks engine will solve small files & optimize write out of the box!

In [0]:
volume_folder = f'/Volumes/{catalog}/{db}/{volume_name}'
def ingest_folder(folder, data_format, table):
  bronze_products = (spark.readStream
                              .format("cloudFiles")
                              .option("cloudFiles.format", data_format)
                              .option("cloudFiles.inferColumnTypes", "true")
                              .option("cloudFiles.schemaLocation", f"{volume_folder}/schema/{table}") #Autoloader will automatically infer all the schema & evolution
                              .load(folder))

  return (bronze_products.writeStream
                    .option("checkpointLocation", f"{volume_folder}/checkpoint/{table}") #exactly once delivery on Delta tables over restart/kill
                    .option("mergeSchema", "true") #merge any new column dynamically
                    .trigger(availableNow= True) #Remove for real time streaming
                    .table("spark_"+table)) #Table will be created if we haven't specified the schema first
  
ingest_folder(f'{volume_folder}/historical_turbine_status', 'json', 'spark_historical_turbine_status')
ingest_folder(f'{volume_folder}/turbine', 'json', 'spark_turbine')
ingest_folder(f'{volume_folder}/incoming_data', 'parquet', 'spark_sensor_bronze').awaitTermination()

In [0]:
%sql 
-- Note the "_rescued_data" column. If we receive wrong data not matching existing schema, it'll be stored here
select * from spark_sensor_bronze;

timestamp,sensor_E,sensor_C,sensor_B,sensor_A,sensor_F,sensor_D,energy,turbine_id,_rescued_data
1705426558,1.9329246538975888,-4.806097405405239,0.3696022660970215,-0.8632605021492814,-2.0313179856870196,3.679920632209072,0.18774784539306655,eaa96f34-d955-be6c-c875-8dcaaa4e4552,null
1705425348,-1.2133433953573696,-1.0332372900745286,-2.122268672431068,-2.492516677870464,-2.569586724056971,-0.39034648081298595,0.00386194541567227,58f7c6fa-71fb-1e66-50d6-6294398299e3,null
1705427308,0.22492983788822563,-5.035925110349594,-5.110384992303199,-0.8502473031892573,-3.292110845355052,0.4328443634593331,0.06552354483242716,fa10dab8-8552-1966-b334-4ea6381cd1a3,null
1705426178,2.2293541086456665,-6.406889948301474,0.23805105457689324,-0.1316408790925363,1.2224881898518187,-3.0055910967216004,0.31474678636123343,cedc547b-cf3d-814a-958b-3dcc0c4ab70c,null
1705427598,-1.7992666697645188,-0.789202780439084,-1.1188272750977135,-0.38120742534632224,-1.8327020706449337,-0.757731168824764,0.7455440217573271,7bbf0ba5-9231-6f40-a69a-353da093fc3c,null
1705426248,5.37269775766045,1.2964146873475522,-1.2280449612859528,-1.5117654270645116,-1.4421054346656579,0.42636998179494734,0.08838428436157696,cbe0db89-46c8-ad8a-eabc-359176928897,null
1705427238,-3.1011324772383437,-3.151370676182969,-1.5813500677100563,-0.10414041020732023,-3.8268932472191537,1.9032296667130901,0.24092456883679358,f48d926a-d687-b5a1-3e63-5dd33926b9a4,null
1705427208,1.3133680645740924,-0.4234991649200198,-2.8136143931921955,-3.1516395256878202,-2.2256002573474523,1.7814349110180971,0.3224403197760138,ece87632-bc98-ff92-02d0-f481d629d440,null
1705426328,3.1389601580589037,1.9068004034158799,2.5494978613885895,-0.38338964374648843,-2.1616964977933626,-3.3309355239543343,0.139916420647844,a950666d-8a49-96ef-b447-c0ceb48438b5,null
1705425978,-0.22433254660107305,1.1357329897106672,-2.7371780888520942,-1.8425934833957731,-4.634707016341732,0.3840583388885357,0.033179827282195255,088b0d61-e9c2-8aa3-c357-882c281618a1,null


In [0]:
%sql 
-- Note the "_rescued_data" column. If we receive wrong data not matching existing schema, it'll be stored here
select * from spark_turbine;

country,lat,location,long,model,state,turbine_id,_rescued_data
US,33.92946,Beaumont,-116.97725,EpicWind,America/Los_Angeles,2a822826-e323-2aff-dd23-9419cf54ea69,null
US,30.16688,Brenham,-96.39774,EpicWind,America/Chicago,78498234-aea4-125d-b1a1-e539b311df9e,null
US,40.81,Morningside Heights,-73.9625,EpicWind,America/New_York,6dbfc0b7-0b16-ab74-674a-828ccefcfe39,null
US,35.05266,Fayetteville,-78.87836,EpicWind,America/New_York,0d65babb-1be1-5175-3d2d-496dd6b07d3a,null
US,42.16808,Huntley,-88.42814,EpicWind,America/Chicago,79827b7a-caea-0518-fd5e-5ee3374cb756,null
US,47.80527,Bothell West,-122.24064,EpicWind,America/Los_Angeles,93e357fa-faf3-9b0a-ca5b-cd5f9d8962a9,null
US,43.54072,Nampa,-116.56346,EpicWind,America/Boise,69ba8009-6e90-6c30-91a6-52f55488c72f,null
US,34.06635,Acworth,-84.67837,EpicWind,America/New_York,60b9d32e-e461-8ed5-ebe3-993a6508fda7,null
US,40.57788,Brighton Beach,-73.95958,EpicWind,America/New_York,a24ada21-bd92-56e2-e265-9107cf84ee19,null
US,42.97086,Port Huron,-82.42491,EpicWind,America/Detroit,847b5bd4-5f91-cc80-cc42-253ce1ced8d9,null


In [0]:
#Let's explore a bit our datasets with pandas on spark.
first_turbine = spark.table('spark_sensor_bronze').limit(1).collect()[0]['turbine_id']
df = spark.table('spark_sensor_bronze').where(f"turbine_id == '{first_turbine}' ").orderBy('timestamp').pandas_api()
df.plot(x="timestamp", y=["sensor_F", "sensor_E"], kind="line")


## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) 2/ Silver data: date cleaned

<img width="700px" style="float:right" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-spark-2.png"/>

We can chain these incremental transformation between tables, consuming only new data.

This can be triggered in near realtime, or in batch fashion, for example as a job running every night to consume daily data.

In [0]:
import pyspark.sql.functions as F
#Compute std and percentil of our timeserie per hour
sensors = [c for c in spark.read.table("spark_sensor_bronze").columns if "sensor" in c]
aggregations = [F.avg("energy").alias("avg_energy")]
for sensor in sensors:
  aggregations.append(F.stddev_pop(sensor).alias("std_"+sensor))
  aggregations.append(F.percentile_approx(sensor, [0.1, 0.3, 0.6, 0.8, 0.95]).alias("percentiles_"+sensor))
  
df = (spark.table("spark_sensor_bronze")
          .withColumn("hourly_timestamp", F.date_trunc("hour", F.from_unixtime("timestamp")))
          .groupBy('hourly_timestamp', 'turbine_id').agg(*aggregations))

df.write.mode('overwrite').saveAsTable("spark_sensor_hourly")
display(spark.table("spark_sensor_hourly"))
#Note: a more scalable solution would be to switch to streaming API and compute the aggregation with a ~3hours watermark and MERGE (upserting) the final output. For this demo clarity we we'll go with a full table update instead.

hourly_timestamp,turbine_id,avg_energy,std_sensor_E,percentiles_sensor_E,std_sensor_C,percentiles_sensor_C,std_sensor_B,percentiles_sensor_B,std_sensor_A,percentiles_sensor_A,std_sensor_F,percentiles_sensor_F,std_sensor_D,percentiles_sensor_D
2024-01-16T17:00:00Z,01833302-b28d-98aa-594d-803700028307,0.33573539509459616,2.736476243024785,"List(-2.0179678499695424, 0.20643568853497743, 1.9428114231167863, 3.402437460099458, 4.86541776070826)",3.373981740861492,"List(-4.477202706775524, -2.3879918797503974, -0.07883938270455815, 1.8523062573117972, 3.880772492597024)",2.1033256042268187,"List(-3.8079028961373727, -2.055745407045629, -0.49250438461725465, 0.9112625964060117, 2.35305795986068)",1.0200520100155366,"List(-2.173215475409898, -1.5045396964960576, -0.7054073625791674, -0.1278658592528894, 0.7314981654601114)",1.897700341413476,"List(-2.630127572304456, -1.597884738694888, -0.11483498122528735, 1.4742719528140886, 2.799164150874028)",2.3087870341911834,"List(-3.3805440427331863, -1.8611615938428487, 0.061634858734052056, 1.5939933191418207, 3.494853515459646)"
2024-01-16T17:00:00Z,02d34789-7bf8-322a-1284-74940587212a,0.32345630477964105,2.4943904793401086,"List(-2.0938725492236636, -0.18024554243855245, 1.6839362286994066, 2.9982405840754294, 4.67466412843168)",3.060198339340619,"List(-4.58849522095522, -2.4166235570168677, 0.0362726999498022, 2.0392169303579015, 3.9163694595034846)",2.0484951935050195,"List(-3.7402458146656365, -2.312749889798754, -0.6151356886668815, 0.7946572460804548, 2.1087738015242556)",1.1399513758292121,"List(-2.2900197016581982, -1.3967899193386621, -0.6865644320687324, -0.11172279013869368, 0.7811134130328263)",1.9833051482145159,"List(-2.670120394292119, -1.6443981608755915, -0.09549406597192847, 1.4270373414372624, 2.757429244860103)",2.3380527374358224,"List(-3.1044765322425056, -1.6689766738543714, 0.23236252775263444, 1.7849113559197765, 3.699332941486324)"
2024-01-16T17:00:00Z,03712b4d-c1d1-0de7-b966-8789e3ec3e3a,0.127877385383332,2.2112699814948424,"List(-1.8720610374581552, -0.07534426323160459, 1.6453470723979717, 3.0359550978552177, 4.542683875688146)",3.3055977383137303,"List(-4.701191180541147, -3.0279292992160634, -0.1697827671386436, 1.4821474763762437, 3.673139394120426)",2.111240092337283,"List(-3.6236513024205026, -2.0870420128716214, -0.4688492036275014, 0.8005891160962768, 2.2168996812703483)",0.9834853693991992,"List(-2.2170220723939282, -1.529492085852846, -0.7944503321239378, -0.185016127216714, 0.6165184940273425)",2.0555969202627105,"List(-2.8851120436056457, -1.7537144483949267, -0.060835193307380964, 1.3697673008074789, 3.1753617303021935)",2.150527867998266,"List(-3.109760510628007, -1.5166130311955242, 0.17801969571960563, 1.7446740460611982, 3.2375404149430453)"
2024-01-16T17:00:00Z,0b2eaa63-5ffb-86c8-769d-d09fb2a724d8,0.10768467007369778,2.314942112769118,"List(-1.692403058658062, 0.007157113254373071, 1.642416662598286, 2.803665519502579, 4.899212868769457)",3.281667326101813,"List(-4.168083063700319, -2.1629581650613368, 0.08921723080661903, 2.209466281481209, 4.304926091906677)",2.2558643455888236,"List(-3.678057137063297, -2.084856451249202, -0.5651799919387104, 0.5876228627224844, 2.468675716996213)",1.028423559724971,"List(-2.319140530665188, -1.5166414428718062, -0.6488937121798568, -0.11402366172279865, 0.7464717966685548)",1.9012987975320315,"List(-2.871154556030271, -1.5677090183469007, -0.15276158969550435, 1.373801874882258, 2.76779584251576)",2.401698398837903,"List(-3.4370536137534264, -1.8724499902645448, 0.016033025559230474, 1.7816140413537478, 3.4475510590711753)"
2024-01-16T17:00:00Z,1bed56a2-90a6-0a71-b546-46535b476d36,0.2782942516934564,2.4060382641357774,"List(-1.951638868132439, 0.020004033362271717, 1.894233250305348, 3.0833960667296036, 5.043106551000316)",3.1222815455423207,"List(-4.510332889305597, -2.1110459856600396, 0.17037031534558622, 2.2486543242749883, 4.198971278329599)",2.0071840587909526,"List(-3.9140172760264065, -2.12435726805


## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) 3/ Build our training dataset

<img width="700px" style="float:right" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-spark-3.png"/>

We can chain these incremental transformation between tables, consuming only new data.

This can be triggered in near realtime, or in batch fashion, for example as a job running every night to consume daily data.

In [0]:
turbine = spark.table("spark_turbine")
health = spark.table("spark_historical_turbine_status")
(spark.table("spark_sensor_hourly")
  .join(turbine, ['turbine_id']).drop("row", "_rescued_data")
  .join(health, ['turbine_id'])
  .drop("_rescued_data")
  .write.mode('overwrite').saveAsTable("spark_turbine_training_dataset"))

display(spark.table("spark_turbine_training_dataset"))

turbine_id,hourly_timestamp,avg_energy,std_sensor_E,percentiles_sensor_E,std_sensor_C,percentiles_sensor_C,std_sensor_B,percentiles_sensor_B,std_sensor_A,percentiles_sensor_A,std_sensor_F,percentiles_sensor_F,std_sensor_D,percentiles_sensor_D,country,lat,location,long,model,state,abnormal_sensor,end_time,maintenance_report,start_time
01833302-b28d-98aa-594d-803700028307,2024-01-16T17:00:00Z,0.33573539509459616,2.736476243024785,"List(-2.0179678499695424, 0.20643568853497743, 1.9428114231167863, 3.402437460099458, 4.86541776070826)",3.373981740861492,"List(-4.477202706775524, -2.3879918797503974, -0.07883938270455815, 1.8523062573117972, 3.880772492597024)",2.1033256042268187,"List(-3.8079028961373727, -2.055745407045629, -0.49250438461725465, 0.9112625964060117, 2.35305795986068)",1.0200520100155366,"List(-2.173215475409898, -1.5045396964960576, -0.7054073625791674, -0.1278658592528894, 0.7314981654601114)",1.897700341413476,"List(-2.630127572304456, -1.597884738694888, -0.11483498122528735, 1.4742719528140886, 2.799164150874028)",2.3087870341911834,"List(-3.3805440427331863, -1.8611615938428487, 0.061634858734052056, 1.5939933191418207, 3.494853515459646)",US,40.5576,Woodbridge,-74.28459,EpicWind,America/New_York,ok,1708017815,null,1705423400
02d34789-7bf8-322a-1284-74940587212a,2024-01-16T17:00:00Z,0.32345630477964105,2.4943904793401086,"List(-2.0938725492236636, -0.18024554243855245, 1.6839362286994066, 2.9982405840754294, 4.67466412843168)",3.060198339340619,"List(-4.58849522095522, -2.4166235570168677, 0.0362726999498022, 2.0392169303579015, 3.9163694595034846)",2.0484951935050195,"List(-3.7402458146656365, -2.312749889798754, -0.6151356886668815, 0.7946572460804548, 2.1087738015242556)",1.1399513758292121,"List(-2.2900197016581982, -1.3967899193386621, -0.6865644320687324, -0.11172279013869368, 0.7811134130328263)",1.9833051482145159,"List(-2.670120394292119, -1.6443981608755915, -0.09549406597192847, 1.4270373414372624, 2.757429244860103)",2.3380527374358224,"List(-3.1044765322425056, -1.6689766738543714, 0.23236252775263444, 1.7849113559197765, 3.699332941486324)",US,29.84576,Estelle,-90.10674,EpicWind,America/Chicago,ok,1708017696,null,1705422613
03712b4d-c1d1-0de7-b966-8789e3ec3e3a,2024-01-16T17:00:00Z,0.127877385383332,2.2112699814948424,"List(-1.8720610374581552, -0.07534426323160459, 1.6453470723979717, 3.0359550978552177, 4.542683875688146)",3.3055977383137303,"List(-4.701191180541147, -3.0279292992160634, -0.1697827671386436, 1.4821474763762437, 3.673139394120426)",2.111240092337283,"List(-3.6236513024205026, -2.0870420128716214, -0.4688492036275014, 0.8005891160962768, 2.2168996812703483)",0.9834853693991992,"List(-2.2170220723939282, -1.529492085852846, -0.7944503321239378, -0.185016127216714, 0.6165184940273425)",2.0555969202627105,"List(-2.8851120436056457, -1.7537144483949267, -0.060835193307380964, 1.3697673008074789, 3.1753617303021935)",2.150527867998266,"List(-3.109760510628007, -1.5166130311955242, 0.17801969571960563, 1.7446740460611982, 3.2375404149430453)",US,40.24537,Pottstown,-75.64963,EpicWind,America/New_York,ok,1708019762,null,1705423112
0b2eaa63-5ffb-86c8-769d-d09fb2a724d8,2024-01-16T17:00:00Z,0.10768467007369778,2.314942112769118,"List(-1.692403058658062, 0.007157113254373071, 1.642416662598286, 2.803665519502579, 4.899212868769457)",3.281667326101813,"List(-4.168083063700319, -2.1629581650613368, 0.08921723080661903, 2.209466281481209, 4.304926091906677)",2.2558643455888236,"List(-3.678057137063297, -2.084856451249202, -0.5651799919387104, 0.5876228627224844, 2.468675716996213)",1.028423559724971,"List(-2.319140530665188, -1.5166414428718062, -0.6488937121798568, -0.11402366172279865, 0.7464717966685548)",1.9012987975320315,"List(-2.871154556030271, -1.5677090183469007, -0.15276158969550435, 1.373801874882258, 2.76779584251576)",2.401698398837903,"List(-3.4370536137534264, -1.8724499902645448, 0.016033025559230474, 1.7816140413537478, 3.4475510590711753)",US,26.68451,Belle Glade,-80.66756,Ep


## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) 4/ Call the ML model and get realtime turbine metrics

<img width="700px" style="float:right" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-spark-4.png"/>

We can chain these incremental transformation between tables, consuming only new data.

This can be triggered in near realtime, or in batch fashion, for example as a job running every night to consume daily data.

In [0]:
import mlflow
mlflow.set_registry_uri('databricks-uc')
#                                                                                                    Stage/version  
#                                                                                       Model name         |        
#                                                                                           |              |        
predict_maintenance = mlflow.pyfunc.spark_udf(spark, f"models:/{catalog}.{db}.dbdemos_turbine_maintenance@prod", "string", env_manager='virtualenv')
#We can use the function in SQL
spark.udf.register("predict_maintenance", predict_maintenance)
columns = predict_maintenance.metadata.get_input_schema().input_names()

2025/10/27 16:28:10 INFO mlflow.pyfunc: This UDF will use virtualenv to recreate the model's software environment for inference. This may take extra time during execution.


2025/10/27 16:28:11 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/10/27 16:28:11 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
2025/10/27 16:29:23 INFO mlflow.utils.virtualenv: Creating a new environment in /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19a26-7df7b-6/mlflow/envs/virtualenv_envs/mlflow-836b50d3da336556c48f20a130dfb71030985159 with /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19a26-7df7b-6/mlflow/envs/pyenv_root/versions/3.12.3/bin/python
2025/10/27 16:29:23 INFO mlflow.utils.virtualenv: Installing dependencies
2025/10/27 16:30:12 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19a26-7df7b-6/mlflow/envs/virtualenv_envs/mlflow-836b50d3da336556c48f20a130dfb71030985159/bin/activate && python -c ""']'


In [0]:
w = Window.partitionBy("turbine_id").orderBy(col("hourly_timestamp").desc())
(spark.table("spark_sensor_hourly")
  .withColumn("row", F.row_number().over(w))
  .filter(col("row") == 1)
  .join(spark.table('spark_turbine'), ['turbine_id']).drop("row", "_rescued_data")
  .withColumn("prediction", predict_maintenance(*columns))
  .write.mode('overwrite').saveAsTable("spark_current_turbine_metrics"))

In [0]:
%sql select * from spark_current_turbine_metrics

turbine_id,hourly_timestamp,avg_energy,std_sensor_E,percentiles_sensor_E,std_sensor_C,percentiles_sensor_C,std_sensor_B,percentiles_sensor_B,std_sensor_A,percentiles_sensor_A,std_sensor_F,percentiles_sensor_F,std_sensor_D,percentiles_sensor_D,country,lat,location,long,model,state,prediction
004a641f-e9e5-9fff-d421-1bf88319420b,2024-01-16T23:00:00Z,0.07481860903879099,2.2120358529793696,"List(-1.963350336964142, 0.8182950089824553, 2.553783167426533, 3.1247732628623863, 5.3566934737771845)",2.8927160852893152,"List(-6.3857155284461715, -4.186826614569899, -0.7317286712910249, 0.43224660135322845, 1.7056106939488793)",2.485293271624966,"List(-3.9242031846007626, -1.220097794546259, -0.30601131669898884, 0.4606103504522574, 1.9445080847223082)",1.058048335093487,"List(-2.0178307013464716, -1.374925451309546, -0.2185303851644379, 0.2203675994158929, 1.0563238136129773)",5.614526027139426,"List(-4.831324461966658, -1.8860077233939423, 0.11300420139128287, 1.5838174616294274, 8.558817961567968)",2.1567050955955853,"List(-3.2828083013404536, -2.08531064166072, -0.5100960390861013, 1.4624090472525024, 3.4597051462261175)",US,34.25807,Tupelo,-88.70464,EpicWind,America/Chicago,ok
00f27248-1f4f-e174-432c-53bd2a9158df,2024-01-16T23:00:00Z,1.443733458624124,1.8343720484469521,"List(-0.7689923219227031, 0.933182372851419, 2.6522561099513053, 3.3352557722580793, 5.11777793466662)",2.7144883867320275,"List(-3.646281225825952, -1.8802905484446897, 0.10327632671140519, 1.5273801103462405, 3.936436407393858)",2.1517912069709655,"List(-4.4405917005211375, -2.487896506919908, -0.6113206456363719, 0.509486077712076, 1.9156142653939643)",1.4395606498822557,"List(-2.6755515985743505, -1.7865377033112493, -0.43933119674576826, 0.1499124428448524, 1.667538062877906)",1.885832738396628,"List(-2.73795166844976, -1.5407595529055957, -0.03729493582098237, 0.7467855724866777, 3.3354101280422945)",2.114420933360022,"List(-3.0831721630606532, -1.6317662704145535, 0.2206106643929654, 1.6153226280379436, 3.30365142073855)",US,42.24113,Crystal Lake,-88.3162,EpicWind,America/Chicago,ok
0102fe7c-5c53-73e0-8af3-36bb0d6ed3c5,2024-01-16T23:00:00Z,0.37892825257836454,1.8393973499561018,"List(-0.6582872164992482, 0.21648554600129422, 1.816513576242951, 3.0804505049008486, 5.124948431167491)",3.1056967718157207,"List(-5.417570105796365, -2.741110330315967, -0.5970388870128314, 1.5497125298526928, 4.890875149011977)",3.6874037276201617,"List(-4.074935479569838, -1.7992670864932612, -0.485483580653602, 1.057783255508217, 7.013167977043018)",1.0103659895975714,"List(-2.5197649236263033, -1.8068336130727802, -1.089145026285636, -0.47910373760264224, 0.6168238235299301)",1.8360190683887692,"List(-3.0363644615259817, -1.7679551476541189, -0.02439292113412239, 1.413160574138546, 2.431889583133381)",2.6203513580781332,"List(-4.081278075807984, -1.803059163026668, 0.4306822047949086, 2.1722479221443614, 5.253714411172801)",US,39.00622,Sterling,-77.4286,EpicWind,America/New_York,ok
01833302-b28d-98aa-594d-803700028307,2024-01-16T23:00:00Z,0.6174721042998128,2.3533096532149167,"List(-1.6797283359608683, 1.1818455011057316, 2.5681953564286664, 3.593617069685049, 6.7804934654902045)",3.464299010586552,"List(-5.004022702016197, -2.5646231898399563, -0.5190024042197406, 0.5957067241046619, 6.7691372141452195)",2.1223303173196295,"List(-4.214981954231753, -2.415333213724523, -1.149419733492103, -0.022753317889737112, 3.9389610412415603)",0.9612457416150144,"List(-2.021605835956015, -1.4471555894272408, -0.7604028811526096, 0.28159350009143824, 0.9149831345071489)",1.830999117441037,"List(-2.7757240165329042, -1.740559073565517, -0.6514136812853988, 0.6241116235656221, 2.867381910577292)",2.56284415031502,"List(-4.102560232072355, -1.4519857137469123, 0.3470461247662504, 1.9174484201810498, 4.301766464530979)",US,40.5576,Woodbridge,-74.28459,EpicWind,America/New_York,ok
027ee813-8748-360f-0165-4e6ebc0f5080,2024-01-16T23:00:00Z,2.3343586848615727,2.2016086339753755,"List(-1.2219977205777

## Simplify your operations with transactional DELETE/UPDATE/MERGE operations

Traditional Data Lake struggle to run these simple DML operations. Using Databricks and Delta Lake, your data is stored on your blob storage with transactional capabilities. You can issue DML operation on Petabyte of data without having to worry about concurrent operations.

In [0]:
spark.sql(f"DELETE FROM spark_sensor_bronze where turbine_id='{first_turbine}'")

DataFrame[num_affected_rows: bigint]

In [0]:
%sql describe history spark_sensor_bronze;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
54,2025-10-27T16:30:29Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(turbine_id#5293 = eaa96f34-d955-be6c-c875-8dcaaa4e4552)""])","List(634400930223470, field-demos_lakehouse-iot-platform, 1041832104988954, 214433439004092, 7644138420879474, manual)",List(2022278721212064),1027-162127-c2xcl9mj,53,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 4, numDeletionVectorsRemoved -> 4, numAddedChangeFiles -> 0, executionTimeMs -> 1976, numDeletionVectorsUpdated -> 4, numDeletedRows -> 2125, scanTimeMs -> 1058, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 908)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
53,2025-10-23T19:52:48Z,7644138420879474,quentin.ambard@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})","List(634400930223470, field-demos_lakehouse-iot-platform, 869013401014819, 13280638276325, 7644138420879474, manual)",List(1364795075128738),1023-194350-7vqgys3j,52,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
52,2025-10-23T19:52:43Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(turbine_id#5293 = 0b2328e0-8c09-eed4-f0ba-edd31057cf6e)""])","List(634400930223470, field-demos_lakehouse-iot-platform, 869013401014819, 13280638276325, 7644138420879474, manual)",List(1364795075128738),1023-194350-7vqgys3j,51,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 4, numDeletionVectorsRemoved -> 4, numAddedChangeFiles -> 0, executionTimeMs -> 2088, numDeletionVectorsUpdated -> 4, numDeletedRows -> 2125, scanTimeMs -> 1110, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 970)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
51,2025-10-23T13:46:00Z,7644138420879474,quentin.ambard@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})","List(634400930223470, field-demos_lakehouse-iot-platform, 143230444529739, 222109739324200, 7644138420879474, manual)",List(2539018256485806),1023-133709-8yi4kct0,50,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
50,2025-10-23T13:45:54Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(turbine_id#5293 = b64e77dd-c8db-d36a-13cb-95e148b85b79)""])","List(634400930223470, field-demos_lakehouse-iot-platform, 143230444529739, 222109739324200, 7644138420879474, manual)",List(2539018256485806),1023-133709-8yi4kct0,49,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 4, numDeletionVectorsRemoved -> 4, numAddedChangeFiles -> 0, executionTimeMs -> 1993, numDeletionVectorsUpdated -> 4, numDeletedRows -> 2125, scanTimeMs -> 1103, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 883)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
49,2025-10-22T14:53:30Z,7644138420879474,quentin.ambard@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})","List(634400930223470, field-demos_lakehouse-iot-platform, 51679912646852, 1427821671020, 7644138420879474, manual)",List(2564664280396682),1022-144419-9pnv4515,48,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
48,2025-10-22T14:53:25Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(turbine_id#5293 = 8bc03cb9-dd08-da09-32a1-8f1b30a67be9)""])","List(634400930223470, field-demos_lakehouse-iot-platform, 51679912646852, 1427821671020, 7644138420879474, manual)",List(2564664280396682),1022-144419-9pnv4515,47,WriteSeri

In [0]:
%sql 
 --also works with AS OF TIMESTAMP "yyyy-MM-dd HH:mm:ss"
select * from spark_sensor_bronze version as of 1 ;

-- You made the DELETE by mistake ? You can easily restore the table at a given version / date:
-- RESTORE TABLE spark_sensor_bronze TO VERSION AS OF 1

-- Or clone it (SHALLOW provides zero copy clone):
-- CREATE TABLE spark_sensor_bronze_clone SHALLOW|DEEP CLONE sensor_bronze VERSION AS OF 1

-- Turn on CDC to capture insert/update/delete operation:
-- ALTER TABLE spark_sensor_bronze SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

timestamp,sensor_E,sensor_C,sensor_B,sensor_A,sensor_F,sensor_D,energy,turbine_id,_rescued_data
1705427048,3.680238993760711,-2.4017933132799176,2.0734829853092407,-1.1915564556400149,2.0482410477610644,-1.9360463128179886,0.9304719240455787,74636c4f-bbdc-f311-5c57-35ea854368b8,null
1705426198,3.312927456066121,-3.2943904381650495,1.7129599873594428,-1.5422393155011818,-0.7113818292743428,-3.2025684973445037,0.3535372117456058,079c4b70-2dc1-c0fd-8d7c-5848a9bb9ca1,null
1705427628,3.1387600562150717,-0.5223851799291372,-4.1865976618062195,-1.133437190821339,0.20705152151896367,0.12016256275126547,0.2906999474631001,aa891d19-a1cc-765c-79a2-84641d0cfa3d,null
1705425258,-4.203939680141756,0.1872185111068283,-5.674889882508366,-1.4180791511545845,1.0796994995124913,3.3468962162759146,0.12286956044787721,610e4ad5-09c4-7055-dff4-948fe6b4f832,null
1705426548,0.9748443997729632,-0.022331386334189407,2.6310923396846304,-1.0619656930712502,0.2670447950523569,0.28720191435832376,0.1266601231499892,2e281650-1188-af2b-9bab-6dd900949476,null
1705425798,1.2175836704395588,2.060671440091906,0.06218858335929811,-0.05922187720398453,-0.1869436889194076,-1.0841291452456507,0.2588960972567357,aca129eb-c4ce-3b2e-809d-0048a10f47a9,null
1705426708,-0.6871497376437823,2.870935531569903,-0.5147379962024177,0.24181346185582409,-0.354497421506643,-0.4001582139744091,0.17513194913188726,177de88e-6b8a-d678-29b4-1769253357a6,null
1705427558,0.9889464457185069,2.2461851444229453,-1.5485414786757334,-1.3835714169352635,-2.072642210843528,-2.588021090654448,0.7127298715753021,375598f8-4414-f8ad-a904-174506a7418d,null
1705425388,2.546391011831106,-3.094224686903196,1.3736642384513544,-0.6439342519580967,-3.5614189701899637,-0.21666308018688618,0.29255703197561245,0e10452c-287e-096f-e23d-f9e44065c2ab,null
1705427538,1.6844202050811858,-2.0732665263393044,1.3606557116176927,-1.0088970294275112,-0.9539893877511418,-2.2764375923992315,0.028686180134204584,4fa9f715-e248-5263-586a-c6018e9f9c9a,null


In [0]:
%sql
--Note: can be turned on by default or for all the database
ALTER TABLE spark_turbine                  SET TBLPROPERTIES (delta.autooptimize.optimizewrite = TRUE, delta.autooptimize.autocompact = TRUE );
ALTER TABLE spark_sensor_bronze            SET TBLPROPERTIES (delta.autooptimize.optimizewrite = TRUE, delta.autooptimize.autocompact = TRUE );
ALTER TABLE spark_current_turbine_metrics  SET TBLPROPERTIES (delta.autooptimize.optimizewrite = TRUE, delta.autooptimize.autocompact = TRUE );

## Our finale tables are now ready to be used to build SQL Dashboards and ML models for predictive maintenance!
<img style="float: right" width="400" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-dashboard-1.png"/>

Switch to Databricks SQL to see how this data can easily be requested with the [Turbine DBSQL Dashboard](/sql/dashboards/a6bb11d9-1024-47df-918d-f47edc92d5f4) to start reviewing our Wind Turbine stats or the [DBSQL Predictive maintenance Dashboard](/sql/dashboards/d966eb63-6d37-4762-b90f-d3a2b51b9ba8).

Creating a single flow was simple.  However, handling many data pipeline at scale can become a real challenge:
* Hard to build and maintain table dependencies 
* Difficult to monitor & enforce advance data quality
* Impossible to trace data lineage
* Difficult pipeline operations (observability, error recovery)


#### To solve these challenges, Databricks introduced **Spark Declarative Pipelines**
A simple way to build and manage data pipelines for fresh, high quality data!

# Next: secure and share data with Unity Catalog

Now that these tables are available in our Lakehouse, let's review how we can share them with the Data Scientists and Data Analysts teams.

Jump to the [Governance with Unity Catalog notebook]($../../02-Data-governance/02-UC-data-governance-security-iot-turbine) or [Go back to the introduction]($../../00-IOT-wind-turbine-introduction-lakehouse)